In [ ]:
import pandas as pd
import numpy as np

In [ ]:
pwd
 

#### 1. Set your working directory and import the dataset Enaho01A-2023-300.csv using Pandas.

In [ ]:
data = pd.read_csv("./Enaho01A-2023-300.csv", encoding="ISO-8859-10")

In [ ]:
data

##### 1.1 Read and display the first 5 rows.

In [ ]:
data.head(5)

##### 1.2 Convert the column names into a list and print it.

In [ ]:
data.columns

In [ ]:
#Printing the list
columns_list = data.columns.tolist()

In [ ]:
#Printing the list
columns_list

##### 1.3 Check the data types of the DataFrame.

In [ ]:
data.dtypes

##### 1.4 Select a subsample containing the variables ['CONGLOME', 'VIVIENDA', 'HOGAR', 'CODPERSO'] and between 3–5 additional variables of your interest.

In [ ]:
sub_data = data[["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO", "P301A", "P302", "P313", "P314A"]]
# P301A: ¿Cuál es el último año o grado de estudios y nivel que aprobó?
# P302: ¿Sabe leer y escribir?
# P313: ¿Cuál es la principal razón por la que no está matriculado o no asiste a algún centro o programa de educación básica o superior?
# P314A: En el mes anterior, ¿Ud. hizo uso del servicio de Internet?

In [ ]:
sub_data

#### 2. Data Manipulation (Data Cleaning):

##### 2.1 Explore the DataFrame using summary functions.

In [ ]:
sub_data.shape

In [ ]:
sub_data.dtypes

In [ ]:
sub_data.describe(include="all")

##### 2.2 Identify if there are missing values.

In [ ]:
sub_data.isnull().sum()

##### 2.3 If they exist, remove them.
There are none in the sub dataframe created

#### 3. Import a second dataset

In [ ]:
data2 = pd.read_csv("./Enaho01a-2023-500.csv", encoding="ISO-8859-10")

In [ ]:
data2

##### 3.1 Display the first 5 rows.

In [ ]:
data2.head(5)

##### 3.2 Convert the column names into a list and print it.

In [ ]:
data2.columns

In [ ]:
#converting the columns into a list
columns_list2 = data2.columns.tolist()

In [ ]:
#Printing the list
columns_list2

##### 3.3 Check the data types.

In [ ]:
data2.dtypes

##### 3.4 Select a subsample containing the variables ['CONGLOME', 'VIVIENDA', 'HOGAR', 'CODPERSO'] and between 3–5 additional variables of your interest.

In [ ]:
sub_data2 = data2[["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO", "P501", "P507", "P513T", "P521C"]]
# P501: La semana pasada, del ... al ..., ¿Tuvo Ud. algún trabajo?
# P507: Ud. se desempeño en su ocupación principal o negocio como:
# P513T: ¿Cuántas horas trabajó la semana pasada, en su ocupación principal, el día: Total
# P521C: ¿Desea ud. otro trabajo y ha hecho algo por cambiar su trabajo actual?

In [ ]:
sub_data2

##### 3.5 Perform the following modifications:

A. Change the data type of a variable (e.g., from text to numeric).

In [ ]:
sub_data2.dtypes

In [ ]:
cols = ["P507", "P513T", "P521C"]
sub_data2.loc[:, cols] = sub_data2[cols].replace(" ", pd.NA)

In [ ]:
print(sub_data2[cols].head())
print(sub_data2[cols].isna().sum())

In [ ]:
sub_data2 = sub_data2.astype({"P507": "Int64", "P513T": "Int64", "P521C": "Int64"})

In [ ]:
sub_data2.dtypes

B. Modify some values in a specific column.

In [ ]:
sub_data2.head(15)

In [ ]:
sub_data2["P507"] = sub_data2["P507"].fillna(0).astype("int64")

In [ ]:
sub_data2.head(15)

#### 4. Merging Datasets:

##### 4.1 Identify the common columns between the two datasets

In [ ]:
common_cols = sub_data.columns.intersection(sub_data2.columns)
print(common_cols)

##### 4.2 Verify whether the values match in both datasets. If not, correct the mismatched values to ensure a proper merge.

In [ ]:
id_cols = ['CONGLOME', 'VIVIENDA', 'HOGAR', 'CODPERSO']

df1_aligned = sub_data.set_index(id_cols)
df2_aligned = sub_data2.set_index(id_cols)

# Filtrar solo las columnas comunes que no son ID
common_cols_no_id = [c for c in df1_aligned.columns if c in df2_aligned.columns]

df1_common = df1_aligned[common_cols_no_id].sort_index().sort_index(axis=1)
df2_common = df2_aligned[common_cols_no_id].sort_index().sort_index(axis=1)

# Reindexar ambos para que tengan exactamente las mismas filas y columnas
df1_common, df2_common = df1_aligned[common_cols_no_id].align(
    df2_aligned[common_cols_no_id],
    join="inner",   # solo las filas que están en ambos
    axis=0
)

# Ahora comparar
mismatches = df1_common != df2_common

print("Número de diferencias por columna:")
print(mismatches.sum())

##### 4.5 Perform the merge.

In [ ]:
id_cols = ['CONGLOME', 'VIVIENDA', 'HOGAR', 'CODPERSO']

# Merge con sufijos para distinguir las columnas que se repiten
merged = sub_data.merge(
    sub_data2,
    on=id_cols,
    how="inner",              # usa "outer" si quieres todas las filas
    suffixes=("_df1", "_df2")
)

merged.shape

##### 4.6 Display the first 5 rows of the resulting DataFrame.

In [ ]:
merged.head(5)

#### 5. In the resulting DataFrame:

##### 5.1 Group the data by a variable of your choice using groupby().

In [ ]:
grouped = merged.groupby("CONGLOME").size().reset_index(name="count")

grouped.head(15)

##### 5.2 Calculate a relevant statistical indicator, for example: the average income per category.

In [ ]:
group_stats = merged.groupby("CONGLOME")["P301A"].agg(
    mean="mean",
    median="median",
    std="std",
    min="min",
    max="max"
).reset_index()

print(group_stats.head())